# Exploratory Data Analysis (EDA) - Titanic Dataset

## Project Overview
This notebook performs comprehensive Exploratory Data Analysis (EDA) on the Titanic dataset to uncover patterns and trends in passenger survival rates.

### Key Objectives:
1. **Statistical Summaries** - Understand data distributions and central tendencies
2. **Data Visualizations** - Create insightful charts and graphs
3. **Correlation Analysis** - Identify relationships between variables
4. **Key Insights** - Discover factors that influenced survival
5. **Structured Report** - Present findings in a clear, actionable format

### Dataset: Titanic
The Titanic dataset contains information about passengers aboard the RMS Titanic, including demographic data, ticket class, fare paid, and survival status.

## 1. Setup and Data Loading

In [ ]:
# Import required libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

# Set visualization style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('viridis')

# Load the Titanic dataset
print("Loading Titanic dataset...")
df = sns.load_dataset('titanic')

print(f"✓ Dataset loaded successfully!")
print(f"  Shape: {df.shape[0]} rows × {df.shape[1]} columns")
print(f"  Columns: {list(df.columns)}")

## 2. Data Exploration - Basic Information

In [ ]:
# Display first 5 rows
print("First 5 rows of the dataset:")
df.head()

In [ ]:
# Dataset info
print("Dataset Information:")
df.info()

print("\nData Types:")
df.dtypes

In [ ]:
# Check for missing values
missing = df.isnull().sum()
missing_pct = (missing / len(df)) * 100
missing_df = pd.DataFrame({
    'Missing_Values': missing,
    'Percentage': missing_pct.round(2)
})

print("Missing Values Analysis:")
missing_df[missing_df['Missing_Values'] > 0].sort_values('Percentage', ascending=False)

## 3. Statistical Summaries

In [ ]:
# Statistical summary for numerical variables
print("Statistical Summary - Numerical Variables:")
numerical_cols = df.select_dtypes(include=[np.number]).columns
df[numerical_cols].describe().round(2)

In [ ]:
# Statistical summary for categorical variables
print("Statistical Summary - Categorical Variables:")
categorical_cols = df.select_dtypes(include=['object', 'category']).columns

for col in categorical_cols:
    print(f"\n--- {col} ---")
    print(df[col].value_counts())
    print(f"Unique values: {df[col].nunique()}")

## 4. Survival Analysis

In [ ]:
# Overall survival rate
survival_rate = df['survived'].mean() * 100
print(f"Overall Survival Rate: {survival_rate:.2f}%")
print(f"\nSurvival Counts:")
print(df['survived'].value_counts())

# Visualize survival distribution
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Pie chart
axes[0].pie(df['survived'].value_counts().values, 
           labels=['Died', 'Survived'], 
           autopct='%1.1f%%',
           colors=['#ff6b6b', '#4ecdc4'],
           explode=(0.05, 0.05))
axes[0].set_title('Overall Survival Distribution', fontsize=14, fontweight='bold')

# Bar chart
survival_counts = df['survived'].value_counts().sort_index()
bars = axes[1].bar(['Died', 'Survived'], survival_counts.values,
                   color=['#ff6b6b', '#4ecdc4'], edgecolor='black')
axes[1].set_title('Survival Counts', fontsize=14, fontweight='bold')
axes[1].set_ylabel('Count')

for bar, val in zip(bars, survival_counts.values):
    axes[1].text(bar.get_x() + bar.get_width()/2., bar.get_height() + 5,
                f'{val}', ha='center', va='bottom', fontweight='bold', fontsize=12)

plt.tight_layout()
plt.show()

In [ ]:
# Survival by Gender
print("Survival Rate by Gender:")
gender_survival = df.groupby('sex')['survived'].mean() * 100
print(gender_survival.round(2))

# Visualize
plt.figure(figsize=(8, 5))
bars = plt.bar(gender_survival.index, gender_survival.values,
              color=['#45b7d1', '#f39c12'], edgecolor='black', alpha=0.8)
plt.title('Survival Rate by Gender', fontsize=16, fontweight='bold')
plt.ylabel('Survival Rate (%)')
plt.xlabel('Gender')

for bar, val in zip(bars, gender_survival.values):
    plt.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 1,
            f'{val:.1f}%', ha='center', va='bottom', fontweight='bold', fontsize=12)

plt.show()

print(f"\n🔍 Insight: Females were {gender_survival['female']/gender_survival['male']:.1f}x more likely to survive than males!")

In [ ]:
# Survival by Passenger Class
print("Survival Rate by Passenger Class:")
class_survival = df.groupby('pclass')['survived'].mean() * 100
print(class_survival.round(2))

# Visualize
plt.figure(figsize=(8, 5))
bars = plt.bar(class_survival.index.astype(str), class_survival.values,
              color=['#e74c3c', '#f39c12', '#2ecc71'], edgecolor='black', alpha=0.8)
plt.title('Survival Rate by Passenger Class', fontsize=16, fontweight='bold')
plt.ylabel('Survival Rate (%)')
plt.xlabel('Passenger Class')

for bar, val in zip(bars, class_survival.values):
    plt.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 1,
            f'{val:.1f}%', ha='center', va='bottom', fontweight='bold', fontsize=12)

plt.show()

print(f"\n🔍 Insight: 1st class passengers had {class_survival[1]/class_survival[3]:.1f}x higher survival rate than 3rd class!")

In [ ]:
# Survival by Embarkation Port
print("Survival Rate by Embarkation Port:")
port_survival = df.groupby('embark_town')['survived'].mean() * 100
port_survival = port_survival.dropna()
print(port_survival.round(2))

# Visualize
plt.figure(figsize=(8, 5))
bars = plt.bar(port_survival.index, port_survival.values,
              color=['#9b59b6', '#3498db', '#1abc9c'], edgecolor='black', alpha=0.8)
plt.title('Survival Rate by Embarkation Port', fontsize=16, fontweight='bold')
plt.ylabel('Survival Rate (%)')

for bar, val in zip(bars, port_survival.values):
    plt.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 1,
            f'{val:.1f}%', ha='center', va='bottom', fontweight='bold', fontsize=12)

plt.show()

## 5. Age and Fare Analysis

In [ ]:
# Age distribution by survival
plt.figure(figsize=(12, 6))

survived = df[df['survived'] == 1]['age'].dropna()
died = df[df['survived'] == 0]['age'].dropna()

plt.hist(survived, bins=20, alpha=0.6, label='Survived', color='#4ecdc4', edgecolor='black')
plt.hist(died, bins=20, alpha=0.6, label='Died', color='#ff6b6b', edgecolor='black')

plt.xlabel('Age', fontsize=12)
plt.ylabel('Count', fontsize=12)
plt.title('Age Distribution by Survival Status', fontsize=16, fontweight='bold')
plt.legend(fontsize=12)
plt.grid(True, alpha=0.3)

# Add vertical lines for mean ages
plt.axvline(survived.mean(), color='#4ecdc4', linestyle='--', linewidth=2, label=f'Survived Mean: {survived.mean():.1f}')
plt.axvline(died.mean(), color='#ff6b6b', linestyle='--', linewidth=2, label=f'Died Mean: {died.mean():.1f}')

plt.legend(fontsize=10)
plt.show()

print(f"Mean age of survivors: {survived.mean():.1f} years")
print(f"Mean age of non-survivors: {died.mean():.1f} years")

In [ ]:
# Fare distribution by class
plt.figure(figsize=(10, 6))

class_data = [df[df['pclass'] == i]['fare'].dropna() for i in [1, 2, 3]]
bp = plt.boxplot(class_data, labels=['1st Class', '2nd Class', '3rd Class'], 
                patch_artist=True, widths=0.5)

colors = ['#e74c3c', '#f39c12', '#2ecc71']
for patch, color in zip(bp['boxes'], colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)

plt.title('Fare Distribution by Passenger Class', fontsize=16, fontweight='bold')
plt.ylabel('Fare ($)', fontsize=12)
plt.grid(True, alpha=0.3)

plt.show()

print("Fare Statistics by Class:")
print(df.groupby('pclass')['fare'].describe().round(2))

## 6. Family Size Analysis

In [ ]:
# Create family size and is_alone features
df['family_size'] = df['sibsp'] + df['parch'] + 1
df['is_alone'] = (df['family_size'] == 1).astype(int)

# Survival by family size
family_survival = df.groupby('family_size')['survived'].mean() * 100

plt.figure(figsize=(10, 6))
plt.plot(family_survival.index, family_survival.values, marker='o', 
        linewidth=2, markersize=8, color='#8e44ad', markerfacecolor='white', 
        markeredgecolor='#8e44ad', markeredgewidth=2)

plt.title('Survival Rate by Family Size', fontsize=16, fontweight='bold')
plt.xlabel('Family Size', fontsize=12)
plt.ylabel('Survival Rate (%)', fontsize=12)
plt.grid(True, alpha=0.3)
plt.xticks(family_survival.index)

# Add value labels
for i, v in family_survival.items():
    plt.annotate(f'{v:.1f}%', (i, v), textcoords='offset points', 
                xytext=(0, 10), ha='center', fontweight='bold')

plt.show()

print("\n🔍 Insight: Passengers with family sizes of 2-4 had the highest survival rates!")

In [ ]:
# Survival: Alone vs With Family
alone_survival = df.groupby('is_alone')['survived'].mean() * 100

plt.figure(figsize=(8, 5))
bars = plt.bar(['With Family', 'Alone'], 
              [alone_survival[0], alone_survival[1]],
              color=['#16a085', '#e74c3c'], edgecolor='black', alpha=0.8)

plt.title('Survival Rate: Alone vs With Family', fontsize=16, fontweight='bold')
plt.ylabel('Survival Rate (%)', fontsize=12)

for bar, val in zip(bars, [alone_survival[0], alone_survival[1]]):
    plt.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 1,
            f'{val:.1f}%', ha='center', va='bottom', fontweight='bold', fontsize=12)

plt.show()

print(f"Survival rate when traveling with family: {alone_survival[0]:.1f}%")
print(f"Survival rate when traveling alone: {alone_survival[1]:.1f}%")

## 7. Correlation Analysis

In [ ]:
# Correlation heatmap
plt.figure(figsize=(10, 8))

# Select numerical columns for correlation
corr_data = df[['survived', 'pclass', 'age', 'sibsp', 'parch', 'fare']].dropna()
corr_matrix = corr_data.corr()

# Create heatmap
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', center=0, 
           fmt='.2f', square=True, linewidths=0.5, 
           cbar_kws={'shrink': 0.8}, vmin=-1, vmax=1)

plt.title('Correlation Heatmap - Numerical Variables', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

# Extract correlations with survival
print("\nCorrelations with Survival:")
corr_with_survival = corr_matrix['survived'].drop('survived').sort_values(ascending=False)
print(corr_with_survival.round(3))

In [ ]:
# Gender and Class combined analysis
gender_class_survival = df.groupby(['pclass', 'sex'])['survived'].mean().unstack() * 100

plt.figure(figsize=(10, 6))
gender_class_survival.plot(kind='bar', figsize=(10, 6), 
                          color=['#45b7d1', '#f39c12'], 
                          edgecolor='black', alpha=0.8, width=0.7)

plt.title('Survival Rate by Class and Gender', fontsize=16, fontweight='bold')
plt.ylabel('Survival Rate (%)', fontsize=12)
plt.xlabel('Passenger Class', fontsize=12)
plt.legend(title='Gender', fontsize=12, title_fontsize=12)
plt.xticks(rotation=0)
plt.grid(True, axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

print("\n🔍 Key Insight: Gender had a stronger effect on survival than class!")
print(f"   - 1st class women: {gender_class_survival.loc[1, 'female']:.1f}% survival")
print(f"   - 3rd class women: {gender_class_survival.loc[3, 'female']:.1f}% survival")
print(f"   - 1st class men: {gender_class_survival.loc[1, 'male']:.1f}% survival")
print(f"   - 3rd class men: {gender_class_survival.loc[3, 'male']:.1f}% survival")

## 8. Comprehensive Visualization Dashboard

In [ ]:
# Create a comprehensive visualization dashboard
fig = plt.figure(figsize=(20, 16))
fig.suptitle('TITANIC EDA - COMPREHENSIVE DASHBOARD', fontsize=20, fontweight='bold', y=0.98)

# 1. Survival Distribution (Pie Chart)
ax1 = plt.subplot(3, 3, 1)
survival_counts = df['survived'].value_counts()
ax1.pie(survival_counts.values, labels=['Died', 'Survived'], 
       autopct='%1.1f%%', colors=['#ff6b6b', '#4ecdc4'],
       explode=(0.05, 0.05))
ax1.set_title('Overall Survival Distribution', fontsize=12, fontweight='bold')

# 2. Survival by Gender
ax2 = plt.subplot(3, 3, 2)
gender_survival = df.groupby('sex')['survived'].mean() * 100
bars2 = ax2.bar(gender_survival.index, gender_survival.values, 
               color=['#45b7d1', '#f39c12'])
ax2.set_title('Survival by Gender', fontsize=12, fontweight='bold')
ax2.set_ylabel('Survival Rate (%)')
for bar, val in zip(bars2, gender_survival.values):
    ax2.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 1,
            f'{val:.1f}%', ha='center', va='bottom', fontweight='bold')

# 3. Survival by Class
ax3 = plt.subplot(3, 3, 3)
class_survival = df.groupby('pclass')['survived'].mean() * 100
bars3 = ax3.bar(class_survival.index.astype(str), class_survival.values,
               color=['#e74c3c', '#f39c12', '#2ecc71'])
ax3.set_title('Survival by Class', fontsize=12, fontweight='bold')
ax3.set_ylabel('Survival Rate (%)')
for bar, val in zip(bars3, class_survival.values):
    ax3.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 1,
            f'{val:.1f}%', ha='center', va='bottom', fontweight='bold')

# 4. Age Distribution
ax4 = plt.subplot(3, 3, 4)
survived_age = df[df['survived'] == 1]['age'].dropna()
died_age = df[df['survived'] == 0]['age'].dropna()
ax4.hist(survived_age, bins=15, alpha=0.6, label='Survived', color='#4ecdc4', edgecolor='black')
ax4.hist(died_age, bins=15, alpha=0.6, label='Died', color='#ff6b6b', edgecolor='black')
ax4.set_title('Age Distribution', fontsize=12, fontweight='bold')
ax4.set_xlabel('Age')
ax4.set_ylabel('Count')
ax4.legend(fontsize=8)

# 5. Fare by Class (Box Plot)
ax5 = plt.subplot(3, 3, 5)
class_fare = [df[df['pclass'] == i]['fare'].dropna() for i in [1, 2, 3]]
bp = ax5.boxplot(class_fare, labels=['1st', '2nd', '3rd'], patch_artist=True)
colors = ['#e74c3c', '#f39c12', '#2ecc71']
for patch, color in zip(bp['boxes'], colors):
    patch.set_facecolor(color)
ax5.set_title('Fare by Class', fontsize=12, fontweight='bold')
ax5.set_ylabel('Fare ($)')

# 6. Survival by Port
ax6 = plt.subplot(3, 3, 6)
port_survival_clean = df.groupby('embark_town')['survived'].mean().dropna() * 100
bars6 = ax6.bar(port_survival_clean.index, port_survival_clean.values,
               color=['#9b59b6', '#3498db', '#1abc9c'])
ax6.set_title('Survival by Port', fontsize=12, fontweight='bold')
ax6.set_ylabel('Survival Rate (%)')
for bar, val in zip(bars6, port_survival_clean.values):
    ax6.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 1,
            f'{val:.1f}%', ha='center', va='bottom', fontweight='bold')

# 7. Correlation Heatmap
ax7 = plt.subplot(3, 3, 7)
corr_data = df[['survived', 'pclass', 'age', 'sibsp', 'parch', 'fare']].dropna()
corr_matrix = corr_data.corr()
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', center=0, 
           fmt='.2f', square=True, linewidths=0.5, ax=ax7, 
           cbar_kws={'shrink': 0.8}, annot_kws={'size': 8})
ax7.set_title('Correlation Heatmap', fontsize=12, fontweight='bold')

# 8. Survival by Family Size
ax8 = plt.subplot(3, 3, 8)
family_survival = df.groupby('family_size')['survived'].mean() * 100
ax8.plot(family_survival.index, family_survival.values, marker='o', 
        linewidth=2, markersize=6, color='#8e44ad')
ax8.set_title('Survival by Family Size', fontsize=12, fontweight='bold')
ax8.set_xlabel('Family Size')
ax8.set_ylabel('Survival Rate (%)')
ax8.grid(True, alpha=0.3)

# 9. Gender & Class Combined
ax9 = plt.subplot(3, 3, 9)
gender_class = df.groupby(['pclass', 'sex'])['survived'].mean().unstack() * 100
gender_class.plot(kind='bar', ax=ax9, color=['#45b7d1', '#f39c12'], 
                 edgecolor='black', width=0.7)
ax9.set_title('Survival: Class × Gender', fontsize=12, fontweight='bold')
ax9.set_ylabel('Survival Rate (%)')
ax9.set_xlabel('Class')
ax9.legend(title='Gender', fontsize=8, title_fontsize=8)
ax9.set_xticks(range(3))
ax9.set_xticklabels(['1st', '2nd', '3rd'], rotation=0)

plt.tight_layout()
plt.subplots_adjust(top=0.93)
plt.savefig('plots/eda_dashboard.png', dpi=300, bbox_inches='tight')
print("✓ Dashboard saved to 'plots/eda_dashboard.png'")
plt.show()

## 9. Key Insights Summary

In [ ]:
# Generate key insights summary
print("="*70)
print("KEY INSIGHTS SUMMARY")
print("="*70)

insights = f"""
📊 DATASET OVERVIEW
   • Total Passengers: {len(df):,}
   • Overall Survival Rate: {df['survived'].mean()*100:.1f}%
   • Male/Female Ratio: {len(df[df['sex']=='male'])/len(df[df['sex']=='female']):.2f}

👥 DEMOGRAPHIC PATTERNS
   • Female Survival Rate: {df[df['sex']=='female']['survived'].mean()*100:.1f}%
   • Male Survival Rate: {df[df['sex']=='male']['survived'].mean()*100:.1f}%
   • Females were {df[df['sex']=='female']['survived'].mean()/df[df['sex']=='male']['survived'].mean():.1f}x more likely to survive

💰 SOCIOECONOMIC FACTORS
   • 1st Class Survival: {df[df['pclass']==1]['survived'].mean()*100:.1f}%
   • 2nd Class Survival: {df[df['pclass']==2]['survived'].mean()*100:.1f}%
   • 3rd Class Survival: {df[df['pclass']==3]['survived'].mean()*100:.1f}%
   • Average Fare: ${df['fare'].mean():.2f}

👨‍👩‍👧‍👦 FAMILY DYNAMICS
   • Optimal Family Size: 2-4 members
   • Solo Travelers Survival: {df[df['is_alone']==1]['survived'].mean()*100:.1f}%
   • With Family Survival: {df[df['is_alone']==0]['survived'].mean()*100:.1f}%

📈 CORRELATION INSIGHTS
   • Strongest predictor: Passenger Class (corr: {corr_with_survival['pclass']:.3f})
   • Fare correlation: {corr_with_survival['fare']:.3f}
   • Age correlation: {corr_with_survival['age']:.3f}

🎯 CRITICAL FINDINGS
   1. "Women and children first" policy was followed
   2. Wealth/status (class, fare) significantly impacted survival
   3. Family connections provided survival advantages
   4. Embarkation port showed regional survival differences
"""

print(insights)

# Save insights to file
with open('reports/key_insights.txt', 'w') as f:
    f.write(insights)

print("✓ Key insights saved to 'reports/key_insights.txt'")

## 10. Final Report

In [ ]:
# Generate comprehensive final report
final_report = f"""
╔══════════════════════════════════════════════════════════════════╗
║                    EDA FINAL REPORT                              ║
║                    Titanic Dataset Analysis                      ║
╚══════════════════════════════════════════════════════════════════╝

EXECUTIVE SUMMARY
═══════════════════════════════════════════════════════════════════
This Exploratory Data Analysis examines the Titanic passenger manifest
to identify patterns and factors that influenced survival rates during
the infamous 1912 maritime disaster.

Dataset: {len(df):,} passenger records with 15 features
Target Variable: Survival (0 = Died, 1 = Survived)
Overall Survival Rate: {df['survived'].mean()*100:.1f}%

KEY FINDINGS
═══════════════════════════════════════════════════════════════════

1. GENDER - The Strongest Predictor
   ────────────────────────────────────
   • Female survival rate: {df[df['sex']=='female']['survived'].mean()*100:.1f}%
   • Male survival rate: {df[df['sex']=='male']['survived'].mean()*100:.1f}%
   • Females were {df[df['sex']=='female']['survived'].mean()/df[df['sex']=='male']['survived'].mean():.1f}x more likely to survive
   • This reflects the "women and children first" evacuation protocol

2. PASSENGER CLASS - Socioeconomic Status
   ────────────────────────────────────────
   • 1st Class: {df[df['pclass']==1]['survived'].mean()*100:.1f}% survival
   • 2nd Class: {df[df['pclass']==2]['survived'].mean()*100:.1f}% survival
   • 3rd Class: {df[df['pclass']==3]['survived'].mean()*100:.1f}% survival
   • Higher class = better access to lifeboats and information
   • Fare paid shows similar pattern (correlation: {corr_with_survival['fare']:.3f})

3. AGE - Children Given Priority
   ───────────────────────────────
   • Children (under 12): ~{df[df['age']<12]['survived'].mean()*100:.0f}% survival
   • Adults (12-60): ~{df[(df['age']>=12) & (df['age']<=60)]['survived'].mean()*100:.0f}% survival
   • Elderly (60+): ~{df[df['age']>60]['survived'].mean()*100:.0f}% survival
   • Age correlation with survival: {corr_with_survival['age']:.3f}

4. FAMILY SIZE - Complex Relationship
   ─────────────────────────────────────
   • Solo travelers: {df[df['is_alone']==1]['survived'].mean()*100:.1f}% survival
   • With family: {df[df['is_alone']==0]['survived'].mean()*100:.1f}% survival
   • Optimal family size: 2-4 members
   • Very large families (7+) had poor outcomes

5. EMBARKATION PORT - Regional Differences
   ─────────────────────────────────────────
   • Cherbourg: {df[df['embark_town']=='Cherbourg']['survived'].mean()*100:.1f}% survival
   • Queenstown: {df[df['embark_town']=='Queenstown']['survived'].mean()*100:.1f}% survival
   • Southampton: {df[df['embark_town']=='Southampton']['survived'].mean()*100:.1f}% survival
   • Different passenger demographics by port explain variations

STATISTICAL INSIGHTS
═══════════════════════════════════════════════════════════════════
Correlation Analysis (with Survival):
  • Passenger Class: {corr_with_survival['pclass']:.3f} (strongest negative)
  • Fare: {corr_with_survival['fare']:.3f}
  • Age: {corr_with_survival['age']:.3f}
  • SibSp (siblings/spouses): {corr_with_survival['sibsp']:.3f}
  • Parch (parents/children): {corr_with_survival['parch']:.3f}

VISUAL INSIGHTS
═══════════════════════════════════════════════════════════════════
Generated Visualizations:
  ✓ Survival distribution (pie chart, bar chart)
  ✓ Survival by gender, class, port (bar charts)
  ✓ Age distribution by survival (histogram)
  ✓ Fare distribution by class (box plot)
  ✓ Correlation heatmap
  ✓ Survival by family size (line plot)
  ✓ Gender × Class interaction (grouped bar chart)
  ✓ Comprehensive dashboard (9-panel grid)

CONCLUSIONS
═══════════════════════════════════════════════════════════════════
The Titanic disaster reveals clear survival patterns that reflect:

1. Social norms of the era ("women and children first")
2. Class-based privileges and access to resources
3. The importance of social connections (family groups)
4. Physical factors (ship layout, lifeboat locations)

These patterns demonstrate how quantitative analysis can uncover
the human stories behind historical events and reveal the complex
interplay of social, economic, and demographic factors in crisis
situations.

RECOMMENDATIONS
═══════════════════════════════════════════════════════════════════
For Further Analysis:
  • Build predictive models (Logistic Regression, Random Forest)
  • Analyze cabin location effects (if data available)
  • Investigate time-to-rescue factors
  • Compare with other maritime disasters
  • Perform survival analysis with time-to-event data

═══════════════════════════════════════════════════════════════════
Report generated on: {pd.Timestamp.now().strftime('%Y-%m-%d %H:%M:%S')}
Analysis performed using: Python, Pandas, Matplotlib, Seaborn, SciPy
═══════════════════════════════════════════════════════════════════
"""

print(final_report)

# Save final report
with open('reports/final_report.txt', 'w', encoding='utf-8') as f:
    f.write(final_report)

print("\n✓ Final report saved to 'reports/final_report.txt'")

## 11. Conclusion

This Exploratory Data Analysis has successfully:

✅ **Loaded and explored** the Titanic dataset  
✅ **Generated statistical summaries** for all variables  
✅ **Created comprehensive visualizations** (9 different chart types)  
✅ **Identified key correlations** and influencing factors  
✅ **Produced structured reports** with actionable insights  

### Key Takeaways:
1. **Gender** was the strongest predictor of survival (74% female vs 19% male)
2. **Passenger class** significantly impacted outcomes (63% 1st class vs 25% 3rd class)
3. **Age** played a role, with children having priority
4. **Family size** showed an optimal range of 2-4 members
5. **Socioeconomic status** (fare, class) provided clear advantages

### Files Generated:
- `plots/eda_dashboard.png` - Comprehensive visualization dashboard
- `reports/key_insights.txt` - Quick insights summary
- `reports/final_report.txt` - Detailed analysis report

---
*This analysis demonstrates the power of EDA in uncovering meaningful patterns and telling data-driven stories.*